# Tutorial: SL-02 Evaluation Toolkit

- Module: 02 Statistical Learning
- Topic: model evaluation and validation workflow
- Estimated runtime: 8-10 minutes
- Prerequisites: empirical risk minimization, linear regression, logistic regression, train/validation/test splits
- Expected outputs: regression metrics, confusion matrix, ROC and PR curves, calibration plot


## Learning goals

After running this notebook, you should be able to:

- apply a consistent train/validation/test split with the shared toolkit;
- evaluate a regression model with MAE, MSE, RMSE, and $R^2$;
- evaluate a classifier with accuracy, precision, recall, F1, ROC-AUC, and PR-AUC;
- read confusion matrices, ROC curves, PR curves, and reliability diagrams; and
- reuse `shared/src/evaluation.py` in later modules.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.linear_model import LinearRegression, LogisticRegression

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.src.evaluation import (
    calibration_curve_data,
    classification_metrics,
    confusion_matrix_report,
    plot_calibration_curve,
    plot_confusion_matrix,
    plot_precision_recall_curve,
    plot_roc_curve,
    precision_recall_curve_data,
    regression_metrics,
    roc_curve_data,
    train_valid_test_split,
)

SEED = 19
np.random.seed(SEED)
SEED


19

## Step 1 - Standardize the split before fitting

A held-out test set is only useful if it is reserved until the end. The shared helper returns train, validation, and test partitions with a single call so later modules can reuse the same workflow.


In [2]:
X_reg, y_reg = make_regression(
    n_samples=180,
    n_features=6,
    n_informative=5,
    noise=14.0,
    random_state=SEED,
)
reg_split = train_valid_test_split(X_reg, y_reg, random_state=SEED)
{key: value.shape for key, value in reg_split.items()}


{'X_train': (108, 6),
 'X_valid': (36, 6),
 'X_test': (36, 6),
 'y_train': (108,),
 'y_valid': (36,),
 'y_test': (36,)}

## Step 2 - Regression example

We start with linear regression so the evaluation logic is separated from classification-specific concepts. The toolkit summarizes four common regression metrics in one place.


In [3]:
regressor = LinearRegression()
regressor.fit(reg_split["X_train"], reg_split["y_train"])

valid_predictions = regressor.predict(reg_split["X_valid"])
test_predictions = regressor.predict(reg_split["X_test"])

regression_summary = {
    "validation": regression_metrics(reg_split["y_valid"], valid_predictions),
    "test": regression_metrics(reg_split["y_test"], test_predictions),
}
regression_summary


{'validation': {'mse': 173.1861050990796,
  'rmse': 13.160019190680522,
  'mae': 10.487145898056301,
  'r2': 0.9919484321347544},
 'test': {'mse': 220.64030522638964,
  'rmse': 14.853965976344151,
  'mae': 11.48300287235226,
  'r2': 0.9850170972201142}}

In [4]:
fig, ax = plt.subplots(figsize=(5.8, 4.2))
ax.scatter(reg_split["y_test"], test_predictions, alpha=0.75, color="tab:blue")
bounds = [min(reg_split["y_test"].min(), test_predictions.min()), max(reg_split["y_test"].max(), test_predictions.max())]
ax.plot(bounds, bounds, linestyle="--", color="tab:gray")
ax.set_title("Regression predictions on the held-out test split")
ax.set_xlabel("True target")
ax.set_ylabel("Predicted target")
fig.tight_layout()
plt.close(fig)
regression_summary["test"]


{'mse': 220.64030522638964,
 'rmse': 14.853965976344151,
 'mae': 11.48300287235226,
 'r2': 0.9850170972201142}

MAE, MSE, and RMSE stay in target units or squared target units, while $R^2$ is unitless. For this synthetic example the linear model is well specified enough that the validation and test summaries are close.


## Step 3 - Classification example

For classification we need both hard predictions and scores. Thresholded predictions drive the confusion matrix, precision, recall, and F1. Probability scores support ROC, PR, and calibration analysis.


In [5]:
X_clf, y_clf = make_classification(
    n_samples=280,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.7, 0.3],
    class_sep=1.15,
    random_state=SEED,
)
clf_split = train_valid_test_split(X_clf, y_clf, random_state=SEED, stratify=y_clf)
classifier = LogisticRegression(max_iter=2000)
classifier.fit(clf_split["X_train"], clf_split["y_train"])

test_prob = classifier.predict_proba(clf_split["X_test"])[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

classification_summary = classification_metrics(
    clf_split["y_test"],
    test_pred,
    y_score=test_prob,
)
classification_summary


{'accuracy': 0.9285714285714286,
 'precision': 0.9333333333333333,
 'recall': 0.8235294117647058,
 'f1': 0.875,
 'roc_auc': 0.9532428355957768,
 'pr_auc': 0.9287238006575829,
 'brier_score': 0.0636213712156889}

## Step 4 - Worked threshold example for ROC and PR

ROC and PR curves come from sweeping a decision threshold across the score vector. The small toy example below makes that mechanism explicit before we look at the model curves.


In [6]:
toy_y = np.array([1, 1, 0, 0, 1, 0])
toy_score = np.array([0.95, 0.8, 0.7, 0.55, 0.4, 0.2])
thresholds = [0.8, 0.55, 0.4]
worked_rows = []

for threshold in thresholds:
    pred = (toy_score >= threshold).astype(int)
    report = confusion_matrix_report(toy_y, pred)["matrix"]
    tn, fp, fn, tp = report.ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    worked_rows.append({
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "recall": round(recall, 3),
        "precision": round(precision, 3),
        "fpr": round(fpr, 3),
    })

worked_rows


[{'threshold': 0.8,
  'tp': 2,
  'fp': 0,
  'fn': 1,
  'tn': 3,
  'recall': np.float64(0.667),
  'precision': np.float64(1.0),
  'fpr': np.float64(0.0)},
 {'threshold': 0.55,
  'tp': 2,
  'fp': 2,
  'fn': 1,
  'tn': 1,
  'recall': np.float64(0.667),
  'precision': np.float64(0.5),
  'fpr': np.float64(0.667)},
 {'threshold': 0.4,
  'tp': 3,
  'fp': 2,
  'fn': 0,
  'tn': 1,
  'recall': np.float64(1.0),
  'precision': np.float64(0.6),
  'fpr': np.float64(0.667)}]

As the threshold falls, recall usually rises because the classifier predicts the positive class more often. Precision may fall if false positives accumulate. ROC plots recall against false positive rate, while PR plots precision against recall.


In [7]:
confusion = confusion_matrix_report(clf_split["y_test"], test_pred)
roc = roc_curve_data(clf_split["y_test"], test_prob)
pr = precision_recall_curve_data(clf_split["y_test"], test_prob)
calibration = calibration_curve_data(clf_split["y_test"], test_prob, n_bins=6)

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.2))
plot_confusion_matrix(confusion, ax=axes[0, 0], title="Confusion Matrix on the Test Split")
plot_roc_curve(roc, ax=axes[0, 1], title="ROC Curve")
plot_precision_recall_curve(
    pr,
    baseline=float(np.mean(clf_split["y_test"])),
    ax=axes[1, 0],
    title="Precision-Recall Curve",
)
plot_calibration_curve(calibration, ax=axes[1, 1], title="Reliability Diagram")
fig.tight_layout()
plt.close(fig)
{
    "confusion_matrix": confusion["matrix"].tolist(),
    "roc_auc": roc.area,
    "pr_auc": pr.area,
    "brier_score": calibration["brier_score"],
}


{'confusion_matrix': [[38, 1], [3, 14]],
 'roc_auc': 0.9532428355957768,
 'pr_auc': 0.9287238006575829,
 'brier_score': 0.0636213712156889}

## Step 5 - Practical reading of the diagnostics

The confusion matrix shows where the thresholded classifier is making mistakes. ROC-AUC summarizes ranking quality across all thresholds, while PR-AUC focuses more directly on positive-class retrieval and is often more informative under class imbalance. The reliability diagram compares predicted probabilities against empirical frequencies, and the Brier score measures probability error directly.


## Summary

This notebook demonstrated one regression model and one classification model with the shared toolkit in `shared/src/evaluation.py`. Later modules can import the same split helper, metric summaries, and plotting functions instead of rebuilding evaluation logic ad hoc.

Next step: pair this notebook with `modules/02-statistical-learning/notes/model-evaluation.md` for the conceptual definitions and workflow guidance.
